# Phase 1 Post-A4 Comparator-Suite Gap Review

Notebook marker: `phase1_post_a4_gap_review_v1_20260420`

Purpose:

- Refresh the Phase 1 governance state after reviewed non-claim A3 and A4 smoke runs.
- Record A2/A2b, A2c, A2d, A3 and A4 as completed **non-claim smoke reviews**.
- Keep headline Phase 1 claims closed because A3/A4 are still smoke proxies and full controls, calibration, influence and reporting are not complete.
- Produce machine-readable blocker artifacts for the next implementation step.

Scientific boundary:

- This notebook does not train models.
- This notebook does not estimate decoder, A3, A4 or privileged-transfer efficacy.
- Smoke metrics from A2/A2b/A2c/A2d/A3/A4 remain implementation diagnostics only.
- A3/A4 smoke review pass does not promote A3/A4 to final claim-bearing comparators.

## Technical basis

V5.5 requires the comparator suite, controls, calibration, influence and reporting package to pass before any headline Phase 1 claim can open. After notebook 13, the project has reviewed smoke artifacts for A3 and A4, but both are explicitly non-claim implementation proxies:

- A3 uses an internal training-scalp-feature teacher/student proxy.
- A4 uses an internal train-time privileged proxy and scalp-only student inference.
- Neither smoke run is final A3/A4 comparator evidence.

This notebook only calls the CLI-backed `phase1_gap_review` and inspects JSON outputs. It must not introduce alternate governance logic.

In [ ]:
# Cell 1 - Runtime, pull latest code, and run tests.

from __future__ import annotations

import base64
import getpass
import hashlib
import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

NOTEBOOK_FIX_MARKER = 'phase1_post_a4_gap_review_v1_20260420'
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')


def _redacted_cmd(cmd):
    redacted = []
    skip_next = False
    for item in cmd:
        if skip_next:
            redacted.append('<redacted>')
            skip_next = False
            continue
        text = str(item)
        if text == '-c':
            redacted.append(text)
            skip_next = True
        elif 'Authorization:' in text or 'github_pat_' in text:
            redacted.append('<redacted>')
        else:
            redacted.append(text)
    return ' '.join(redacted)


def run(cmd, cwd=None, check=True, capture=False):
    print('$', _redacted_cmd([str(x) for x in cmd]))
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
    )
    if capture:
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr, file=sys.stderr)
    if check and completed.returncode:
        raise subprocess.CalledProcessError(completed.returncode, cmd, output=completed.stdout, stderr=completed.stderr)
    return completed


if not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    header = f'Authorization: Basic {auth}'
    run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    print('Repo already exists:', REPO_DIR)
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=False)
if unit_result.returncode != 0:
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])
print('Unit tests passed. Continue to post-A4 gap review.')

In [ ]:
# Cell 2 - Explicit source-of-truth paths.
# These reviewed run paths are intentionally fixed; avoid silently following latest.txt for claim-affecting provenance.

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

PREREG_BUNDLE = ARTIFACT_ROOT / 'prereg' / '20260418T161442014597Z' / 'prereg_bundle.json'
EXPECTED_LOCKED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PHASE1_READINESS_RUN = ARTIFACT_ROOT / 'phase1_readiness' / '20260419T154005857077Z'

A2_A2B_REVIEWED_RUN = ARTIFACT_ROOT / 'phase1_model_smoke' / '20260419T172746816598Z'
A2C_REVIEWED_RUN = ARTIFACT_ROOT / 'phase1_a2c_smoke' / '20260420T081425018260Z'
A2D_REVIEWED_RUN = ARTIFACT_ROOT / 'phase1_a2d_smoke' / '20260420T074006419555Z'
A3_REVIEWED_RUN = ARTIFACT_ROOT / 'phase1_a3_smoke' / '20260420T092747979646Z'
A4_REVIEWED_RUN = ARTIFACT_ROOT / 'phase1_a4_smoke' / '20260420T094721187268Z'

GAP_REVIEW_OUTPUT_ROOT = ARTIFACT_ROOT / 'phase1_gap_review'

for label, path in {
    'PREREG_BUNDLE': PREREG_BUNDLE,
    'PHASE1_READINESS_RUN': PHASE1_READINESS_RUN,
    'A2_A2B_REVIEWED_RUN': A2_A2B_REVIEWED_RUN,
    'A2C_REVIEWED_RUN': A2C_REVIEWED_RUN,
    'A2D_REVIEWED_RUN': A2D_REVIEWED_RUN,
    'A3_REVIEWED_RUN': A3_REVIEWED_RUN,
    'A4_REVIEWED_RUN': A4_REVIEWED_RUN,
}.items():
    print(label, 'exists =', path.exists(), '|', path)

In [ ]:
# Cell 3 - Audit helpers.

def utc_stamp() -> str:
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')


def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def latest_run(root: Path) -> Path | None:
    latest = root / 'latest.txt'
    if latest.exists():
        target = Path(latest.read_text(encoding='utf-8').strip())
        if target.exists():
            return target
    if not root.exists():
        return None
    runs = sorted([p for p in root.iterdir() if p.is_dir()])
    return runs[-1] if runs else None

In [ ]:
# Cell 4 - Validate locked prereg identity and all completed non-claim smoke review notes.

bundle = read_json(PREREG_BUNDLE)
locked_identity_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_file_sha256 = sha256_file(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', locked_identity_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)
assert bundle.get('status') == 'locked'
assert locked_identity_hash == EXPECTED_LOCKED_PREREG_IDENTITY_HASH

review_notes = {
    'A2/A2b': (A2_A2B_REVIEWED_RUN / 'phase1_a2_a2b_model_smoke_review_note.json', 'phase1_a2_a2b_model_smoke_review_pass_non_claim'),
    'A2c': (A2C_REVIEWED_RUN / 'phase1_a2c_coral_smoke_review_note.json', 'phase1_a2c_coral_smoke_review_pass_non_claim'),
    'A2d': (A2D_REVIEWED_RUN / 'phase1_a2d_riemannian_smoke_review_note.json', 'phase1_a2d_riemannian_smoke_review_pass_non_claim'),
    'A3': (A3_REVIEWED_RUN / 'phase1_a3_distillation_smoke_review_note.json', 'phase1_a3_distillation_smoke_review_pass_non_claim'),
    'A4': (A4_REVIEWED_RUN / 'phase1_a4_privileged_smoke_review_note.json', 'phase1_a4_privileged_smoke_review_pass_non_claim'),
}
for label, (path, expected) in review_notes.items():
    print(label, 'review note exists =', path.exists(), '|', path)
    assert path.exists(), f'Missing review note: {path}'
    note = read_json(path)
    print(label, 'status =', note.get('status'))
    assert note.get('status') == expected
print('All reviewed smoke sources are present and remain non-claim.')

In [ ]:
# Cell 5 - Hash the post-A4 gap-review implementation surface.

tracked_sources = [
    REPO_DIR / 'src' / 'phase1' / 'gap_review.py',
    REPO_DIR / 'src' / 'cli.py',
    REPO_DIR / 'tests' / 'unit' / 'test_phase1_gap_review.py',
    REPO_DIR / 'configs' / 'models' / 'distill_a3.yaml',
    REPO_DIR / 'configs' / 'models' / 'privileged_a4.yaml',
    REPO_DIR / 'configs' / 'controls' / 'control_suite_spec.yaml',
    REPO_DIR / 'configs' / 'eval' / 'claim_mapping.yaml',
    REPO_DIR / 'configs' / 'eval' / 'metrics.yaml',
    REPO_DIR / 'configs' / 'eval' / 'inference_defaults.yaml',
    REPO_DIR / 'notebooks' / '14_colab_phase1_post_a4_gap_review.ipynb',
]
source_hashes = {}
for path in tracked_sources:
    source_hashes[str(path.relative_to(REPO_DIR))] = {
        'exists': path.exists(),
        'sha256': sha256_file(path) if path.exists() else None,
    }
print(json.dumps(source_hashes, indent=2))
assert all(item['exists'] for item in source_hashes.values())

In [ ]:
# Cell 6 - Run the CLI-backed post-A4 gap review. No training is launched.

cmd = [
    'python', '-m', 'src.cli', 'phase1_gap_review',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--readiness-run', str(PHASE1_READINESS_RUN),
    '--output-root', str(GAP_REVIEW_OUTPUT_ROOT),
    '--a2-a2b-run', str(A2_A2B_REVIEWED_RUN),
    '--a2c-run', str(A2C_REVIEWED_RUN),
    '--a2d-run', str(A2D_REVIEWED_RUN),
    '--a3-run', str(A3_REVIEWED_RUN),
    '--a4-run', str(A4_REVIEWED_RUN),
]
run(cmd, cwd=REPO_DIR)
print('Post-A4 gap review command completed. Review generated blockers before any claim-bearing work.')

In [ ]:
# Cell 7 - Closeout summary and claim-state assertions.

gap_run = latest_run(GAP_REVIEW_OUTPUT_ROOT)
print('================ Phase 1 Post-A4 Gap Review Closeout ================')
print('Notebook fix marker:', NOTEBOOK_FIX_MARKER)
print('Latest gap review output:', gap_run)
print('Locked prereg hash from bundle:', locked_identity_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)

expected_files = [
    'phase1_comparator_suite_gap_review_inputs.json',
    'phase1_comparator_suite_gap_review_summary.json',
    'phase1_comparator_suite_gap_review_report.md',
    'comparator_suite_status.json',
    'claim_readiness_blockers.json',
    'implementation_backlog.json',
]
for name in expected_files:
    print(name, 'exists =', bool(gap_run and (gap_run / name).exists()))

summary = read_json(gap_run / 'phase1_comparator_suite_gap_review_summary.json')
blockers = read_json(gap_run / 'claim_readiness_blockers.json')
backlog = read_json(gap_run / 'implementation_backlog.json')
status = read_json(gap_run / 'comparator_suite_status.json')

expected_completed = {'A2_A2b', 'A2c_CORAL', 'A2d_riemannian', 'A3_distillation', 'A4_privileged'}
completed = set(summary.get('completed_non_claim_smoke_reviews', []))
expected_blockers = {
    'a3_a4_final_comparator_configs_or_runners_missing',
    'phase1_control_claim_metric_inference_surfaces_still_draft',
    'phase1_final_runner_control_calibration_influence_modules_missing',
    'headline_claim_blocked_until_full_comparator_suite_controls_calibration_influence_reporting_pass',
}
observed_blockers = set(summary.get('blockers', []))
claim_state = blockers.get('claim_state', {})

assert expected_completed.issubset(completed), f'Missing completed smoke reviews: {sorted(expected_completed - completed)}'
assert 'required_non_claim_smoke_reviews_not_all_passed' not in observed_blockers, 'All required non-claim smoke reviews should be present after A4.'
assert expected_blockers.issubset(observed_blockers), f'Missing expected blockers: {sorted(expected_blockers - observed_blockers)}'
assert summary.get('claim_ready') is False
assert summary.get('headline_phase1_claim_open') is False
assert claim_state.get('full_phase1_claim_bearing_run_allowed') is False
assert claim_state.get('headline_phase1_claim_open') is False

draft_surfaces = {item.get('surface') for item in summary.get('draft_governance_surfaces', [])}
assert {'controls', 'claim_mapping', 'metrics', 'inference'}.issubset(draft_surfaces), f'Draft governance surfaces incomplete: {sorted(draft_surfaces)}'

print(json.dumps({
    'status': summary.get('status'),
    'claim_ready': summary.get('claim_ready'),
    'headline_phase1_claim_open': summary.get('headline_phase1_claim_open'),
    'completed_non_claim_smoke_reviews': summary.get('completed_non_claim_smoke_reviews'),
    'blockers': summary.get('blockers'),
    'next_step': summary.get('next_step'),
}, indent=2, ensure_ascii=False))
print()
print('Draft governance surfaces:')
print(json.dumps(summary.get('draft_governance_surfaces'), indent=2, ensure_ascii=False))
print()
print('Missing source modules:')
print(json.dumps(summary.get('missing_source_modules'), indent=2, ensure_ascii=False))
print()
print('Claim state:')
print(json.dumps(claim_state, indent=2, ensure_ascii=False))
print()
print('Next engineering items:')
for item in backlog.get('next_engineering_items', []):
    print('-', item)

print()
print('NOT OK TO CLAIM: This post-A4 gap review does not prove decoder efficacy, A3/A4 efficacy, A4 superiority, or privileged-transfer efficacy.')